# Daily inventory

Workspaces, semantic models, sizes and reports, appended daily to Delta tables.
The same list functions I use interactively, now as a monitoring pipeline.

Size numbers are only real for Import models. Direct Lake reports resident
columns only, so those numbers say more about query activity than model size.

In [1]:
# Setup: only sempy is needed here (built into Fabric), we only read metadata.
# The target lakehouse holds the history; a dedicated monitoring lakehouse works too.

from datetime import datetime, timezone
import pandas as pd
import sempy.fabric as fabric

TARGET_LAKEHOUSE = "LH_Contoso"
SNAPSHOT = datetime.now(timezone.utc).isoformat()

def short_error(e):
    # Keep only the first line: the full XMLA stack trace is noise (and shows your account).
    return str(e).splitlines()[0][:120]

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 45, Finished, Available, Finished, False)

## Collect

One loop over the workspaces. System workspaces without XMLA access
get skipped and logged, the run continues.

In [2]:
# One pass over every workspace you can see: the workspace itself, its models
# (with size where possible) and its reports. Errors are logged, never fatal,
# so one locked workspace doesn't stop the daily run.

ws_rows, sm_rows, rpt_rows = [], [], []

for _, ws in fabric.list_workspaces().iterrows():
    ws_name, ws_id = ws["Name"], ws["Id"]
    ws_rows.append({
        "SnapshotTime": SNAPSHOT,
        "WorkspaceId": ws_id,
        "WorkspaceName": ws_name,
        "CapacityId": ws.get("Capacity Id"),
    })

    try:
        for _, ds in fabric.list_datasets(workspace=ws_name).iterrows():
            row = {
                "SnapshotTime": SNAPSHOT,
                "WorkspaceName": ws_name,
                "Dataset": ds["Dataset Name"],
                "DatasetId": ds.get("Dataset ID"),
                "TotalSizeMB": None,
                "ColumnCount": None,
                "SizeNote": None,
            }
            try:
                df_cols = fabric.list_columns(ds["Dataset Name"], workspace=ws_name, extended=True)
                row["TotalSizeMB"] = round(df_cols["Total Size"].sum() / 1024**2, 1)
                row["ColumnCount"] = len(df_cols)
            except Exception as e:
                row["SizeNote"] = short_error(e)
            sm_rows.append(row)
    except Exception as e:
        print(f"Models skipped for {ws_name}: {short_error(e)}")

    try:
        for _, r in fabric.list_reports(workspace=ws_name).iterrows():
            rpt_rows.append({
                "SnapshotTime": SNAPSHOT,
                "WorkspaceName": ws_name,
                "Report": r["Name"],
                "ReportId": r.get("Id"),
                "DatasetId": r.get("Dataset Id"),
            })
    except Exception as e:
        print(f"Reports skipped for {ws_name}: {short_error(e)}")

print(f"Workspaces: {len(ws_rows)}, Models: {len(sm_rows)}, Reports: {len(rpt_rows)}")

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 46, Finished, Available, Finished, False)

Models skipped for Admin monitoring: The '<euii>peter@dutchbigeek.nl</euii>' user does not have permission to call the Discover method.
Models skipped for Poc : Internal error CapacityNotActive.Capacity b6251990-4049-4c35-be1b-7b20a92d5129 is not active
Models skipped for SpaceParts: Internal error CapacityNotActive.Capacity b6251990-4049-4c35-be1b-7b20a92d5129 is not active
Models skipped for Dutch BI Geek - Fabric - DEV: Internal error CapacityNotActive.Capacity b6251990-4049-4c35-be1b-7b20a92d5129 is not active
Models skipped for X_Lease: The '<euii>peter@dutchbigeek.nl</euii>' user does not have permission to call the Discover method.
Models skipped for Microsoft Fabric Capacity Metrics: The '<euii>peter@dutchbigeek.nl</euii>' user does not have permission to call the Discover method.
Workspaces: 16, Models: 13, Reports: 15


## Write

Append with a snapshot time, so history builds up.
Columns that are entirely None get forced to string — Spark turns them
into a void type otherwise and Delta silently drops them. Cost me an hour.

In [3]:
# Append each snapshot to its Delta table, so every run adds a day of history.
# All-None columns are forced to string, otherwise Delta drops them and the schema drifts.

for rows, table in [
    (ws_rows, "InventoryWorkspaces"),
    (sm_rows, "InventorySemanticModels"),
    (rpt_rows, "InventoryReports"),
]:
    if not rows:
        continue
    pdf = pd.DataFrame(rows)
    for c in pdf.columns:
        if pdf[c].isna().all():
            pdf[c] = pdf[c].astype("string")
    spark.createDataFrame(pdf).write.mode("append").saveAsTable(f"{TARGET_LAKEHOUSE}.dbo.{table}")
    print(f"Appended {len(rows)} rows to {table}")

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 47, Finished, Available, Finished, False)

Appended 16 rows to InventoryWorkspaces
Appended 13 rows to InventorySemanticModels
Appended 15 rows to InventoryReports


## Latest snapshot

Stored in UTC, displayed in local time. Filter on the latest snapshot,
otherwise every model shows up once per day.

In [4]:
# Show only the most recent run, in Dutch time: the table holds every day since the start.

from pyspark.sql import functions as F

df_sm = spark.table(f"{TARGET_LAKEHOUSE}.dbo.InventorySemanticModels")
latest = df_sm.agg(F.max("SnapshotTime")).collect()[0][0]

display(
    df_sm.filter(F.col("SnapshotTime") == latest)
         .withColumn("SnapshotTimeNL", F.from_utc_timestamp(F.to_timestamp("SnapshotTime"), "Europe/Amsterdam"))
         .drop("SnapshotTime")
         .orderBy("WorkspaceName", "Dataset")
)

StatementMeta(, 626535a0-e4d5-4ec1-889f-1b89ae4e045a, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a2f7e7d-c99e-4513-b650-2458011437c2)